# Urban Network Analysis with OSMnx

This notebook analyzes urban street networks and amenities for **Semarang**, **Bandung**, and **Jakarta** using OSMnx.

**Reference:** Boeing, G. 2017. "OSMnx: New Methods for Acquiring, Constructing, Analyzing, and Visualizing Complex Street Networks." *Computers, Environment and Urban Systems* 65, 126-139.

**Analysis includes:**
1. Street network acquisition and visualization
2. Network topology and statistics
3. Centrality analysis (betweenness, closeness)
4. Network connectivity and resilience
5. Amenity distribution analysis
6. Integration with traffic congestion data

## 1. Setup and Installation

In [ ]:
# Install OSMnx if needed (uncomment to run)
# !pip install osmnx networkx geopandas matplotlib contextily

In [ ]:
# Import libraries
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from shapely.geometry import Point, box
import warnings
warnings.filterwarnings('ignore')

# Configure OSMnx
ox.settings.use_cache = True
ox.settings.log_console = False

print(f"OSMnx version: {ox.__version__}")
print(f"NetworkX version: {nx.__version__}")

In [ ]:
# Define study areas (bounding boxes from traffic data)
CITIES = {
    'Semarang': {
        'bbox': (-7.105, -6.919, 110.227, 110.528),  # (south, north, west, east)
        'center': (-7.012, 110.378),
        'color': '#2ecc71'
    },
    'Bandung': {
        'bbox': (-7.0848, -6.8294, 107.4688, 107.8261),
        'center': (-6.957, 107.647),
        'color': '#3498db'
    },
    'Jakarta': {
        'bbox': (-6.4096, -6.0911, 106.6036, 107.11),
        'center': (-6.250, 106.857),
        'color': '#e74c3c'
    }
}

print("Study areas defined!")

## 2. Download Street Networks

In [ ]:
# Download street network for Semarang (smallest, for quick testing)
print("Downloading Semarang street network...")
G_smg = ox.graph_from_bbox(
    bbox=CITIES['Semarang']['bbox'],
    network_type='drive',  # Only drivable roads
    simplify=True
)
print(f"Semarang: {len(G_smg.nodes)} nodes, {len(G_smg.edges)} edges")

In [ ]:
# Download Bandung network
print("Downloading Bandung street network...")
G_bdg = ox.graph_from_bbox(
    bbox=CITIES['Bandung']['bbox'],
    network_type='drive',
    simplify=True
)
print(f"Bandung: {len(G_bdg.nodes)} nodes, {len(G_bdg.edges)} edges")

In [ ]:
# Download Jakarta network (larger, may take longer)
print("Downloading Jakarta street network...")
G_jkt = ox.graph_from_bbox(
    bbox=CITIES['Jakarta']['bbox'],
    network_type='drive',
    simplify=True
)
print(f"Jakarta: {len(G_jkt.nodes)} nodes, {len(G_jkt.edges)} edges")

In [ ]:
# Store networks in dictionary
networks = {
    'Semarang': G_smg,
    'Bandung': G_bdg,
    'Jakarta': G_jkt
}

print("All networks downloaded!")

## 3. Visualize Street Networks

In [ ]:
# Plot all three networks
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (city, G) in zip(axes, networks.items()):
    ox.plot_graph(G, ax=ax, node_size=0, edge_linewidth=0.3,
                  edge_color=CITIES[city]['color'], bgcolor='white',
                  show=False, close=False)
    ax.set_title(f"{city}\n({len(G.nodes):,} nodes, {len(G.edges):,} edges)",
                 fontsize=12, fontweight='bold')

plt.suptitle('Street Networks from OpenStreetMap', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/osm_street_networks.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 4. Basic Network Statistics

In [ ]:
# Calculate basic stats for each network
def get_basic_stats(G, city_name):
    """Calculate basic network statistics"""
    stats = ox.basic_stats(G)
    
    # Add area calculation
    bbox = CITIES[city_name]['bbox']
    # Approximate area in km²
    lat_diff = abs(bbox[1] - bbox[0])
    lon_diff = abs(bbox[3] - bbox[2])
    area_km2 = lat_diff * 111 * lon_diff * 111 * np.cos(np.radians(np.mean([bbox[0], bbox[1]])))
    stats['area_km2'] = area_km2
    
    return stats

# Calculate stats for all cities
all_stats = {}
for city, G in networks.items():
    all_stats[city] = get_basic_stats(G, city)
    print(f"\n{city} Statistics:")
    print(f"  Nodes: {all_stats[city]['n']:,}")
    print(f"  Edges: {all_stats[city]['m']:,}")
    print(f"  Area: {all_stats[city]['area_km2']:.1f} km²")
    print(f"  Street density: {all_stats[city]['street_density_km']:.2f} km/km²")
    print(f"  Node density: {all_stats[city]['node_density_km']:.2f} nodes/km²")
    print(f"  Edge density: {all_stats[city]['edge_density_km']:.2f} edges/km²")
    print(f"  Avg street length: {all_stats[city]['street_length_avg']:.1f} m")
    print(f"  Total street length: {all_stats[city]['street_length_total']/1000:.1f} km")

In [ ]:
# Create comparison table
stats_df = pd.DataFrame({
    'Metric': ['Nodes', 'Edges', 'Area (km²)', 'Street Density (km/km²)', 
               'Node Density (per km²)', 'Avg Street Length (m)', 'Total Street Length (km)'],
    'Semarang': [
        f"{all_stats['Semarang']['n']:,}",
        f"{all_stats['Semarang']['m']:,}",
        f"{all_stats['Semarang']['area_km2']:.1f}",
        f"{all_stats['Semarang']['street_density_km']:.2f}",
        f"{all_stats['Semarang']['node_density_km']:.1f}",
        f"{all_stats['Semarang']['street_length_avg']:.1f}",
        f"{all_stats['Semarang']['street_length_total']/1000:.1f}"
    ],
    'Bandung': [
        f"{all_stats['Bandung']['n']:,}",
        f"{all_stats['Bandung']['m']:,}",
        f"{all_stats['Bandung']['area_km2']:.1f}",
        f"{all_stats['Bandung']['street_density_km']:.2f}",
        f"{all_stats['Bandung']['node_density_km']:.1f}",
        f"{all_stats['Bandung']['street_length_avg']:.1f}",
        f"{all_stats['Bandung']['street_length_total']/1000:.1f}"
    ],
    'Jakarta': [
        f"{all_stats['Jakarta']['n']:,}",
        f"{all_stats['Jakarta']['m']:,}",
        f"{all_stats['Jakarta']['area_km2']:.1f}",
        f"{all_stats['Jakarta']['street_density_km']:.2f}",
        f"{all_stats['Jakarta']['node_density_km']:.1f}",
        f"{all_stats['Jakarta']['street_length_avg']:.1f}",
        f"{all_stats['Jakarta']['street_length_total']/1000:.1f}"
    ]
})

stats_df

## 5. Network Topology Analysis

In [ ]:
# Analyze node degrees (street connectivity)
def analyze_node_degrees(G, city_name):
    """Analyze the distribution of node degrees (number of streets meeting at intersections)"""
    # Get degree of each node
    degrees = dict(G.degree())
    degree_counts = pd.Series(degrees).value_counts().sort_index()
    
    # Calculate statistics
    degree_values = list(degrees.values())
    stats = {
        'mean_degree': np.mean(degree_values),
        'median_degree': np.median(degree_values),
        'max_degree': max(degree_values),
        'dead_ends': sum(1 for d in degree_values if d == 1),
        'three_way': sum(1 for d in degree_values if d == 3),
        'four_way': sum(1 for d in degree_values if d == 4),
        'degree_counts': degree_counts
    }
    
    return stats

# Analyze all cities
degree_stats = {}
for city, G in networks.items():
    degree_stats[city] = analyze_node_degrees(G, city)
    print(f"\n{city} Node Degree Analysis:")
    print(f"  Mean degree: {degree_stats[city]['mean_degree']:.2f}")
    print(f"  Dead ends (degree=1): {degree_stats[city]['dead_ends']:,}")
    print(f"  3-way intersections: {degree_stats[city]['three_way']:,}")
    print(f"  4-way intersections: {degree_stats[city]['four_way']:,}")

In [ ]:
# Plot node degree distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (city, stats) in zip(axes, degree_stats.items()):
    counts = stats['degree_counts']
    ax.bar(counts.index, counts.values, color=CITIES[city]['color'], alpha=0.7, edgecolor='black')
    ax.axvline(stats['mean_degree'], color='red', linestyle='--', label=f"Mean: {stats['mean_degree']:.2f}")
    ax.set_xlabel('Node Degree (connections)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'{city}', fontsize=12, fontweight='bold')
    ax.legend()
    ax.set_xlim(0, 8)

plt.suptitle('Node Degree Distribution (Street Connectivity)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/osm_node_degree_distribution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 6. Centrality Analysis

In [ ]:
# Calculate betweenness centrality for Semarang (smaller network for faster computation)
print("Calculating betweenness centrality for Semarang...")
print("(This may take a few minutes for larger networks)")

# Use edge betweenness centrality
edge_bc_smg = nx.edge_betweenness_centrality(G_smg, normalized=True)
nx.set_edge_attributes(G_smg, edge_bc_smg, 'betweenness')

print(f"Done! Max betweenness: {max(edge_bc_smg.values()):.6f}")

In [ ]:
# Plot betweenness centrality for Semarang
fig, ax = plt.subplots(figsize=(12, 10))

# Get edge colors based on betweenness
edge_colors = [edge_bc_smg.get((u, v), edge_bc_smg.get((v, u), 0)) for u, v, k in G_smg.edges(keys=True)]

ox.plot_graph(G_smg, ax=ax, node_size=0, edge_linewidth=1,
              edge_color=edge_colors, edge_cmap=plt.cm.hot_r,
              bgcolor='white', show=False, close=False)

ax.set_title('Semarang - Edge Betweenness Centrality\n(Critical routes for network flow)',
             fontsize=14, fontweight='bold')

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=plt.cm.hot_r, norm=plt.Normalize(vmin=min(edge_colors), vmax=max(edge_colors)))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.7)
cbar.set_label('Betweenness Centrality', fontsize=11)

plt.savefig('figures/osm_betweenness_semarang.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Calculate closeness centrality for nodes
print("Calculating closeness centrality...")

# For Semarang
closeness_smg = nx.closeness_centrality(G_smg)
nx.set_node_attributes(G_smg, closeness_smg, 'closeness')

print(f"Mean closeness: {np.mean(list(closeness_smg.values())):.6f}")
print(f"Max closeness: {max(closeness_smg.values()):.6f}")

In [ ]:
# Plot closeness centrality
fig, ax = plt.subplots(figsize=(12, 10))

node_colors = [closeness_smg[node] for node in G_smg.nodes()]

ox.plot_graph(G_smg, ax=ax, node_size=5, node_color=node_colors,
              node_cmap=plt.cm.viridis, edge_linewidth=0.2, edge_color='gray',
              bgcolor='white', show=False, close=False)

ax.set_title('Semarang - Closeness Centrality\n(Accessibility to all other nodes)',
             fontsize=14, fontweight='bold')

sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=plt.Normalize(vmin=min(node_colors), vmax=max(node_colors)))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.7)
cbar.set_label('Closeness Centrality', fontsize=11)

plt.savefig('figures/osm_closeness_semarang.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 7. Street Orientation Analysis

In [ ]:
# Plot street orientations (polar histograms)
fig, axes = plt.subplots(1, 3, figsize=(15, 5), subplot_kw={'projection': 'polar'})

for ax, (city, G) in zip(axes, networks.items()):
    # Get bearings
    G_bearing = ox.bearing.add_edge_bearings(G)
    bearings = pd.Series([d['bearing'] for u, v, k, d in G_bearing.edges(keys=True, data=True)])
    
    # Create histogram
    n_bins = 36
    bins = np.linspace(0, 360, n_bins + 1)
    counts, _ = np.histogram(bearings, bins=bins)
    
    # Convert to radians
    angles = np.deg2rad(bins[:-1])
    width = np.deg2rad(360 / n_bins)
    
    ax.bar(angles, counts, width=width, color=CITIES[city]['color'], alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_title(f'{city}', fontsize=12, fontweight='bold', pad=10)

plt.suptitle('Street Orientation (Polar Histogram)', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('figures/osm_street_orientation.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 8. Amenity Analysis

In [ ]:
# Download amenities for Semarang
print("Downloading amenities for Semarang...")

# Define amenity tags to download
amenity_tags = {
    'amenity': ['hospital', 'school', 'university', 'restaurant', 'bank', 'fuel', 'parking'],
    'shop': True,
    'tourism': ['hotel', 'museum']
}

bbox = CITIES['Semarang']['bbox']
amenities_smg = ox.features_from_bbox(
    bbox=bbox,
    tags={'amenity': True}
)

print(f"Found {len(amenities_smg)} amenities in Semarang")

In [ ]:
# Count amenity types
if 'amenity' in amenities_smg.columns:
    amenity_counts = amenities_smg['amenity'].value_counts().head(20)
    print("Top 20 Amenity Types in Semarang:")
    print(amenity_counts)

In [ ]:
# Plot amenity distribution
fig, ax = plt.subplots(figsize=(12, 8))

if len(amenity_counts) > 0:
    amenity_counts.head(15).plot(kind='barh', ax=ax, color='#2ecc71', edgecolor='black')
    ax.set_xlabel('Count', fontsize=12)
    ax.set_ylabel('Amenity Type', fontsize=12)
    ax.set_title('Semarang - Top 15 Amenity Types', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
plt.tight_layout()
plt.savefig('figures/osm_amenities_semarang.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Map amenities on street network
fig, ax = plt.subplots(figsize=(14, 12))

# Plot street network
ox.plot_graph(G_smg, ax=ax, node_size=0, edge_linewidth=0.3, edge_color='gray',
              bgcolor='white', show=False, close=False)

# Plot amenities (only points, not polygons)
amenities_points = amenities_smg[amenities_smg.geometry.type == 'Point']
if len(amenities_points) > 0:
    amenities_points.plot(ax=ax, color='red', markersize=5, alpha=0.5, label='Amenities')

ax.set_title('Semarang - Street Network with Amenities', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')

plt.savefig('figures/osm_network_amenities_semarang.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 9. Integration with Traffic Data

In [ ]:
# Load traffic data
print("Loading traffic data...")

traffic_smg = gpd.read_file('traffic_smg_output/evening_peak_smg.gpkg')
traffic_bdg = gpd.read_file('traffic_bdg_output/evening_peak_bdg.gpkg')
traffic_jkt = gpd.read_file('traffic_jkt_output/evening_peak_jkt.gpkg')

print(f"Semarang traffic segments: {len(traffic_smg)}")
print(f"Bandung traffic segments: {len(traffic_bdg)}")
print(f"Jakarta traffic segments: {len(traffic_jkt)}")

In [ ]:
# Convert OSM network to GeoDataFrame for comparison
edges_smg = ox.graph_to_gdfs(G_smg, nodes=False)
nodes_smg = ox.graph_to_gdfs(G_smg, edges=False)

print(f"OSM edges: {len(edges_smg)}")
print(f"Traffic segments: {len(traffic_smg)}")

In [ ]:
# Overlay traffic data on OSM network
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: OSM network
ax1 = axes[0]
edges_smg.plot(ax=ax1, color='gray', linewidth=0.3)
ax1.set_title('OSM Street Network\n(OpenStreetMap)', fontsize=12, fontweight='bold')
ax1.set_axis_off()

# Right: Traffic data
ax2 = axes[1]
traffic_smg.plot(column='jam_factor_mean', cmap='RdYlGn_r', linewidth=0.5, ax=ax2,
                 legend=True, legend_kwds={'label': 'Jam Factor', 'shrink': 0.7})
ax2.set_title('HERE Traffic Data\n(Evening Peak)', fontsize=12, fontweight='bold')
ax2.set_axis_off()

plt.suptitle('Semarang - OSM Network vs Traffic Data Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/osm_vs_traffic_comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 10. Network Efficiency Analysis

In [ ]:
# Calculate network efficiency metrics
def calculate_efficiency_metrics(G, city_name):
    """Calculate network efficiency and connectivity metrics"""
    
    # Convert to undirected for some metrics
    G_undir = G.to_undirected()
    
    metrics = {}
    
    # Average clustering coefficient
    metrics['avg_clustering'] = nx.average_clustering(G_undir)
    
    # Check if graph is connected
    if nx.is_connected(G_undir):
        metrics['is_connected'] = True
        metrics['diameter'] = nx.diameter(G_undir)
        metrics['avg_path_length'] = nx.average_shortest_path_length(G_undir)
    else:
        metrics['is_connected'] = False
        # Get largest connected component
        largest_cc = max(nx.connected_components(G_undir), key=len)
        G_largest = G_undir.subgraph(largest_cc).copy()
        metrics['largest_component_size'] = len(largest_cc)
        metrics['largest_component_pct'] = len(largest_cc) / len(G_undir.nodes()) * 100
        if len(G_largest) > 1:
            metrics['diameter'] = nx.diameter(G_largest)
            # Sample for large graphs
            if len(G_largest) > 500:
                sample_nodes = list(G_largest.nodes())[:500]
                G_sample = G_largest.subgraph(sample_nodes)
                metrics['avg_path_length'] = nx.average_shortest_path_length(G_sample)
            else:
                metrics['avg_path_length'] = nx.average_shortest_path_length(G_largest)
    
    return metrics

print("Calculating efficiency metrics (this may take a while)...")
efficiency_smg = calculate_efficiency_metrics(G_smg, 'Semarang')

print(f"\nSemarang Network Efficiency:")
print(f"  Connected: {efficiency_smg.get('is_connected', 'N/A')}")
print(f"  Avg Clustering Coefficient: {efficiency_smg['avg_clustering']:.4f}")
if 'diameter' in efficiency_smg:
    print(f"  Network Diameter: {efficiency_smg['diameter']}")
if 'avg_path_length' in efficiency_smg:
    print(f"  Avg Path Length: {efficiency_smg['avg_path_length']:.2f}")
if 'largest_component_pct' in efficiency_smg:
    print(f"  Largest Component: {efficiency_smg['largest_component_pct']:.1f}%")

## 11. Export Results

In [ ]:
# Export network statistics
export_stats = []
for city, G in networks.items():
    stats = all_stats[city]
    export_stats.append({
        'City': city,
        'Nodes': stats['n'],
        'Edges': stats['m'],
        'Area_km2': round(stats['area_km2'], 2),
        'Street_Density_km_per_km2': round(stats['street_density_km'], 2),
        'Node_Density_per_km2': round(stats['node_density_km'], 2),
        'Avg_Street_Length_m': round(stats['street_length_avg'], 2),
        'Total_Street_Length_km': round(stats['street_length_total']/1000, 2),
        'Mean_Node_Degree': round(degree_stats[city]['mean_degree'], 2),
        'Dead_Ends': degree_stats[city]['dead_ends'],
        'Three_Way_Intersections': degree_stats[city]['three_way'],
        'Four_Way_Intersections': degree_stats[city]['four_way']
    })

export_df = pd.DataFrame(export_stats)
export_df.to_csv('osm_network_statistics.csv', index=False)
print("Saved: osm_network_statistics.csv")
export_df

In [ ]:
# Save networks as GeoPackage
print("Saving networks as GeoPackage...")

for city, G in networks.items():
    # Convert to GeoDataFrames
    nodes, edges = ox.graph_to_gdfs(G)
    
    # Save edges
    filename = f"osm_network_{city.lower()}.gpkg"
    edges.to_file(filename, driver='GPKG', layer='edges')
    nodes.to_file(filename, driver='GPKG', layer='nodes')
    print(f"  Saved: {filename}")

print("\nDone!")

## Summary

This notebook demonstrated OSMnx analysis including:

1. **Network Acquisition** - Downloaded street networks from OpenStreetMap
2. **Basic Statistics** - Nodes, edges, street density, lengths
3. **Topology Analysis** - Node degrees, intersection types
4. **Centrality Analysis** - Betweenness and closeness centrality
5. **Street Orientation** - Polar histogram of street bearings
6. **Amenity Analysis** - POI distribution
7. **Traffic Integration** - Comparison with HERE traffic data
8. **Efficiency Metrics** - Clustering, diameter, path length

**Reference:**
Boeing, G. 2017. "OSMnx: New Methods for Acquiring, Constructing, Analyzing, and Visualizing Complex Street Networks." *Computers, Environment and Urban Systems* 65, 126-139. https://geoffboeing.com/publications/osmnx-paper/